# <img style="float: left; padding-right: 20px; width: 200px" src="https://raw.githubusercontent.com/raxlab/imt2200-data/main/media/logo.jpg">  IMT 2200 - Introducción a Ciencia de Datos
**Pontificia Universidad Católica de Chile**<br>
**Instituto de Ingeniería Matemática y Computacional**<br>
**Semestre 2025-S2**<br>
**Profesor:** Rodrigo A. Carrasco <br>

# <h1><center>Actividad 04: Obteniendo Datos de la Web</center></h1>

Esta actividad busca aplicar conocimientos sobre lectura de datos desde la web en distintos formatos (scrapping y APIs) para la creación de un dataset unificado.

## Instrucciones

Este Notebook contiene las instrucciones a realizar para la actividad. 

<b>Al finalizarla, deben subir el Notebook y los archivos generados en un único archivo .zip, al módulo de la Actividad 04 en Canvas. Entregas posteriores al cierre de la actividad serán evaluadas con nota 1.0.</b>

## Actividad

Para esta actividad, queremos analizar la calidad del aire de las ciudades más pobladas del mundo. Para esto, realice los siguientes pasos:

**1. Extraer datos con web scrapping y API**

Vamos a extraer una lista de las ciudades más pobladas desde Wikipedia, específicamente en el siguiente URL:

`URL_1 = https://en.wikipedia.org/wiki/List_of_largest_cities#List`

Por otra parte, usaremos la [Open-Meteo API](https://open-meteo.com/). Este es un servicio open-source de meteorología que nos permitirá obtener datos como las coordenadas de una ciudad y sus parámetros climáticos, como la calidad del aire.

* 1.1 Utilizando las librerías `requests`y `BeautifulSoup`, obtenga todas las filas y columnas de la tabla de Wikipedia de las ciudades más grandes del mundo y genere un DataFrame a partir de ellas. Su DataFrame debe contener como mínimo las siguientes columnas: ciudad, país y población estimada.

* 1.2 Transforme la columna de población en valores numéricos y sólo deje las 20 mayores ciudades.

**2. Llamada a la API**

* 2.1 Ahora, utilizando `requests`, haga un llamado al siguiente URL de la API de Open-Meteo, reemplazando el valor `CIUDAD` con cada uno de los nombres de las ciudades de su DataFrame:

`URL_2 = https://geocoding-api.open-meteo.com/v1/search?name={CIUDAD}&count=1&language=en&format=json`

Haga una copia de su DataFrame anterior. En esta copia, agregue dos columnas nuevas y guarde los valores obtenidos de latitud y longitud (sin modificar el DataFrame original).

* 2.2 Con los datos de las coordenadas, podemos acceder a información sobre la calidad del aire actual disponible con Open-Meteo. Nuevamente, para todas las ciudades, utilice el URL dado para obtener el índices de calidad del aire (usaremos el europeo) y la cantidad de partículas en suspensión.

`URL_3 = https://air-quality-api.open-meteo.com/v1/air-quality?latitude={LAT}&longitude={LON}&current=european_aqi,pm10,pm2_5`

Guarde los valores obtenidos en nuevas columnas del mismo DataFrame.

* 2.3 En la documentación de Open-Meteo ([aquí](https://open-meteo.com/en/docs/air-quality-api)), podemos ver el significado de los valores del índice European AQI. Utilizando la función `aqi2str()` entregada, genere una nueva columna `Air Quality` (string) a partir de los valores que obtuvo mediante la API.

* 2.4 Revise los valores obtenidos. ¿Tienen sentido? Si hay valores que considere inválidos o "outliers" (extremadamente altos), descártelos del dataset.

**3. Visualización**

Vamos a generar dos visualizaciones a partir de las ciudades con las que hemos trabajado. Para esto, usaremos una nueva librería de visualización llamada `plotly.express`. Plotly permite generar gráficos interactivos, con tooltips donde podemos mostrar información adicional de nuestro DataFrame, lo cual los hace muy convenientes para la exploración de un dataset.

Lea y complete el código entregado con los valores de su DataFrame. Ejecute las celdas y responda:

* Entre las ciudades más pobladas, ¿cuál es la calidad de aire más común?

* ¿Cómo es la relación entre tamaño de población y calidad del aire de las ciudades?

* ¿Hay algún lugar del mundo donde se vea una mayor concentración de grandes ciudades? Si la hay, ¿cómo es la calidad del aire en estas zonas?

## Rúbrica

- Si han hecho todo y sólo hay errores menores: 7.0
- Si sólo llegaron hasta la parte 2.1: 5.0
- Menos que eso: 1.0

### 0. Algunas librerías

Las siguientes son algunas de las librerías que recomendamos usar para esta Actividad. Puede agregar más si lo requiere.

In [1]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
from io import StringIO

### 1. Extraer datos

#### 1.1 Respuesta:

In [2]:
# 1.1
url = "https://en.wikipedia.org/wiki/List_of_largest_cities#List"
headers = {'User-Agent': 'maripiu'}
page = requests.get(url, headers=headers, timeout=10)
print(page.status_code)

200


In [4]:
datos = pd.read_html(StringIO(page.text))
datos[1]

City[a]        Country UN 2018 population estimates[b]  \
        City[a]        Country UN 2018 population estimates[b]   
0         Tokyo          Japan                        37468000   
1         Delhi          India                        28514000   
2      Shanghai          China                        25582000   
3     São Paulo         Brazil                        21650000   
4   Mexico City         Mexico                        21581000   
..          ...            ...                             ...   
76   Washington  United States                         5207000   
77       Yangon        Myanmar                         5157000   
78   Alexandria          Egypt                         5086000   
79        Jinan          China                         5052000   
80  Guadalajara         Mexico                         5023000   

                City proper[c]                                             \
                    Definition      Population Area (km2)  Density (/km2)   
0        Metropolis prefecture        13515271       2191      6,169 [13]   
1   National Capital Territory        16753235       1484     11,289 [15]   
2                 Municipality        24870895       6341  3,922 [17][18]   
3                 Municipality        12252023       1521      8,055 [19]   
4                   City-state         9209944       1485      6,202 [21]   
..                         ...             ...        ...             ...   
76            Federal district          702455        177      3,969 [29]   
77                        City  4,728,524[103]          —               —   
78           Urban governorate         5441866       2300      2,366[104]   
79       City (sub-provincial)         9202432      10244    898 [17][18]   
80                Municipality         1385621        151      9,176 [21]   

   Urban area[12]                           Metropolitan area[d]             \
       Population Area (km2) Density (/km2)           Population Area (km2)   
0        37785000       8231      4,591 [e]             37274000      13452   
1        32226000       2344     13,748 [f]             29000000       3483   
2        24073000       4333      5,556 [g]                    —          —   
3        23086000       3649      6,327 [h]             21734682       7947   
4        21804000       2530           8618             21804515       7866   
..            ...        ...            ...                  ...        ...   
76        7631000       5501      1,387 [t]              6263245      17009   
77        6874000        666          10321                    —          —   
78        4712000        293          16082                    —          —   
79        4017000        932           4310                    —          —   
80        5525000        816           6771              5286642       3560   

                   
   Density (/km2)  
0      2,771 [14]  
1      8,326 [16]  
2               —  
3      2,735 [20]  
4      2,772 [22]  
..            ...  
76      368 [102]  
77              —  
78              —  
79              —  
80           1485  

[81 rows x 13 columns]

#### 1.2 Respuesta:

In [ ]:
# 1.2 transforme la columna de población en valores numéricos y sólo deje las 20 mayores ciudades.
soup = bs(page.text)
tables = soup.find_all('table')
len(tables)
tables[1]

In [ ]:
df = pd.DataFrame(columns = ['City','Country','Population city','pounds'])

# iterar sobre cada fila ('tr') para completar la información
for row in tables[1].find_all('tr')[1::]:
    cols = row.find_all("td")
    if len(cols) > 0:
        #print(cols)
        cols = [col.text.strip() for col in cols]
        #print(cols)
        City = cols[1]
        Country = cols[2]
        Population = cols[3]
        new_row = pd.DataFrame({'City': City, 'Country': Country, 'Population City': Population, index=['country'])
        df = pd.concat([df, new_row], ignore_index=True)
df.head()

### 2. Uso de API

#### 2.1 Respuesta:

In [ ]:
# 2.1
url2 = "https://geocoding-api.open-meteo.com/v1/search?name={CIUDAD}&count=1&language=en&format=json"
headers2 = {'User-Agent': 'maripiu'}
respuesta = requests.get(url2, headers=headers2, timeout=10)
print(page.status_code)

#### 2.2 Respuesta:

In [ ]:
# 2.2

#### 2.3 Respuesta:

In [ ]:
# ==== CODIGO ENTREGADO - NO MODIFICAR ====
air_quality = {
    "Good": [0, 20],
    "Fair": [20, 40],
    "Moderate": [40, 60],
    "Poor": [60, 80],
    "Very Poor": [80, 100],
    "Extremely Poor": [100, float('inf')]
}

def aqi2str(aqi):
    for key, (low, high) in air_quality.items():
        if low <= aqi < high:
            return key
    return "Unknown"

In [ ]:
# 2.3

#### 2.4 Respuesta:

In [ ]:
# 2.4

### 3. Visualizar datos

* ¿Cómo son los valores de calidad de aire para las ciudades más pobladas? ¿Cuál es lo más común?

* ¿Cómo es la relación entre tamaño de población y calidad del aire de una ciudad?

* ¿Hay algún lugar del mundo donde se vea una mayor concentración de grandes ciudades? Si la hay, ¿cómo es la calidad del aire?

In [ ]:
# Figura 1: Barplot de calidad del aire
import plotly.express as px

by_quality = new_df.groupby('Air Quality').size().reset_index(name='Count')

fig = px.bar(by_quality,
            x='Air Quality',
            y='Count',
            title="Calidad del aire de 80 ciudades más pobladas del mundo",
            labels={
                "Count": "Cantidad de ciudades",
                "Air Quality": "Calidad del aire"
            },
            color='Air Quality')

fig.update_layout(
    height=400,
    width=900,
)
fig.update_xaxes(categoryorder='array',
                 categoryarray= ["Good", "Fair", "Moderate", "Poor", "Very Poor", "Extremely Poor"]
)
fig.show()

#### Respuesta:

In [ ]:
# Figura 2: Scattermap entre población y calidad del aire

fig = px.scatter(df_filtered,
                 x="Population",
                 y="eu_aqi",
                 title="Relación entre población y calidad del aire",
                 labels={
                     "Population": "Población",
                     "eu_aqi": "Calidad del aire (EU AQI)"
                    },
                    hover_data={
                        "City": True,
                        "Country": True,
                        "Population": True,
                        "eu_aqi": True,
                        "Air Quality": True
                        }
                )

fig.update_layout(
    height=500,
    width=900,
)

#### Respuesta:

In [ ]:
# Figura 3: Mapa mundial de ciudades más pobladas

fig = px.scatter_geo(data_frame=df_filtered, # Su dataframe
                    lat='lat', # Columna de latitud
                    lon='lon', # Columna de longitud
                    color='eu_aqi', # Columna que representa el color de los puntos
                    hover_name='City', # Columna para el titulo del tooltip
                    projection="natural earth",
                    color_continuous_scale=px.colors.sequential.Inferno_r,
                    title="Calidad del aire de ciudades más pobladas", # Titulo del grafico
                    hover_data={
                        # Qué columnas mostrar en el tooltip
                        "Country": True,
                        "eu_aqi": True,
                        "Air Quality": True
                        # Puede agregar otras...
                    },
                )

fig.update_layout(
    margin={"r":0,"t":50,"l":0,"b":0}, # Márgenes del gráfico
    height=600, # Altura del gráfico
    width=800, # Ancho del gráfico
)
fig.update_traces(
    marker=dict(size=10), # Tamaño de los puntos
)
fig.show()

#### Respuesta: